In [2]:
# Import dependencies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Define the file path
file_path_ratings = '/content/drive/MyDrive/Colab Notebooks/ratings.csv'

# Load data
ratings = pd.read_csv(file_path_ratings)
ratings.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


**a) Generate user-item matrix for ratings dataframe**

In [4]:
from scipy.sparse import csr_matrix
# Function: create_user_item_matrix
# Description: To generate a user_item matrix with users as rows and movies as columns from the ratings dataframe
def create_user_item_matrix(df):
  '''
    Args:
        df: pandas dataframe containing 3 columns (userId, movieId, rating)

    Returns:
        X: sparse matrix
        user_mapper: dict that maps user id's to user indices
        user_inv_mapper: dict that maps user indices to user id's
        movie_mapper: dict that maps movie id's to movie indices
        movie_inv_mapper: dict that maps movie indices to movie id's
  '''

  M = df['userId'].nunique()
  N = df['movieId'].nunique()

  user_mapper = dict(zip(np.unique(df["userId"]), list(range(M))))
  movie_mapper = dict(zip(np.unique(df["movieId"]), list(range(N))))
  user_inv_mapper = dict(zip(list(range(M)), np.unique(df['userId'])))
  movie_inv_mapper = dict(zip(list(range(N)), np.unique(df['movieId'])))

  user_index = [user_mapper[i] for i in df['userId']]
  movie_index = [movie_mapper[i] for i in df['movieId']]

  X = csr_matrix((df["rating"], (user_index, movie_index)), shape=(M, N))
  return X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper


In [5]:
X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_user_item_matrix(ratings)
X.shape

(610, 9724)

**b) Calculation of sparsity of the user-item matrix**

In [6]:
# Calculate sparsity of the user-item matrix
n_total = X.shape[0]*X.shape[1]
n_ratings = X.nnz
n_empty = n_total - n_ratings
sparsity = n_empty/n_total
print(f"Matrix sparsity: {round(sparsity*100,2)}%")

Matrix sparsity: 98.3%


Matrix sparsity is the measure of how many elements in the matrix are zero. In a user-item matrix, it tells us how many items have not been rated by the users.

Matrix sparsity of 98.3% indicates that ratings dataframe has 1.7% of ratings stored and the rest of the cells are empty.

**c) Item Recommendations with k-Nearest Neighbours**

In [7]:
from sklearn.neighbors import NearestNeighbors

# Function: recommend_movies
# Description: to recommend top 5 movies for the user based on their rating history
def recommend_movies(movie_id, X, movie_mapper, movie_inv_mapper, k=5, metric='cosine'):
  """
    Finds k-nearest neighbours for a given movie id.

    Args:
        movie_id: id of the movie of interest
        X: user-item utility matrix
        k: number of similar movies to retrieve
        metric: distance metric for kNN calculations

    Output: returns list of k similar movie ID's
    """

  X = X.T
  neighbour_ids = []

  movie_ind = movie_mapper[movie_id]
  movie_vec = X[movie_ind]
  if isinstance(movie_vec, (np.ndarray)):
      movie_vec = movie_vec.reshape(1,-1)
  kNN = NearestNeighbors(n_neighbors=k+1, algorithm="brute", metric=metric)
  kNN.fit(X)
  neighbour = kNN.kneighbors(movie_vec, return_distance=False)
  # print('Neighbor', neighbour[:5])
  for i in range(0,k):
      n = neighbour.item(i)
      neighbour_ids.append(int(movie_inv_mapper[n]))
  neighbour_ids.pop(0)
  return neighbour_ids

In [23]:
similar_movies = recommend_movies(1, X, movie_mapper, movie_inv_mapper, k=5)
similar_movies

[3114, 480, 780, 260]

**d) Function to handle cold-start problem for new users**

In [9]:
from collections import defaultdict

# Function: new_user_recommendations
# Description: to recommend movies for new users based on the ratings user has already provided
def new_user_recommendations(user_ratings, X, movie_mapper, movie_inv_mapper, k=5, metric='cosine'):
  '''
    Produces a list of new user recommendations based on
    similarity to previously rated items.

    Args:
        user_ratings: dictionary of the user's ratings
        X: user-item utility matrix
        k: number of similar movies to retrieve
        metric: distance metric for kNN calculations

    Output: returns list of k similar movie ID
  '''
  scores = defaultdict(float)
  for movie_id, rating in user_ratings.items():
    # Check if the movie_id exists in the movie_mapper before proceeding
    if movie_id in movie_mapper:
      similar = recommend_movies(movie_id, X, movie_mapper, movie_inv_mapper, k=k, metric=metric)
      # Weigh each similar movie with the ratings provided by the user for the target movie Id
      for sim_movie in similar:
        scores[sim_movie] += rating
    else:
      print(f"Warning: Movie ID {movie_id} not found in the dataset. Skipping this movie.")

  print('Scores: ', scores )

  # Exclude movies the user has already rated
  for rated_movie_id in user_ratings:
      scores.pop(rated_movie_id, None)

  # Rank by aggregated score, descending
  ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
  return [movie_id for movie_id, _ in ranked]

# Sample data which was collected during registration of the user
user_ratings = {1:5, 480:4, 3114:4, 340:3}
recommendations = new_user_recommendations(user_ratings, X, movie_mapper, movie_inv_mapper, k=5)
recommendations

Scores:  defaultdict(<class 'float'>, {3114: 5.0, 480: 5.0, 780: 5.0, 260: 5.0, 589: 4.0, 356: 4.0, 110: 4.0, 457: 4.0, 2355: 4.0, 1: 4.0, 4306: 4.0, 4886: 4.0, 352: 3.0, 179: 3.0, 291: 3.0, 209: 3.0})


[780, 260, 589, 356, 110, 457, 2355, 4306, 4886, 352, 179, 291, 209]

**Evaluation of the recommender system using Mean Squared Error**

In [10]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

R_train, R_test = train_test_split(ratings, test_size=0.2, random_state=42)
print(f"R_train shape: {R_train.shape}")
print(f"R_test shape: {R_test.shape}")

R_train shape: (80668, 4)
R_test shape: (20168, 4)


In [12]:
# Dimensionality reduction with Matrix Factorization
from sklearn.decomposition import TruncatedSVD

# Rebuild the user-item matrix on the training data
X_train, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_user_item_matrix(R_train)
print(f"X_train shape: {X_train.shape}")

# Fit SVD on the training matrix
svd = TruncatedSVD(n_components=20, random_state=42)
user_vector = svd.fit_transform(X_train)
item_vector = svd.components_.T

# Reconstruct the user-item matrix with predicted ratings
predicted_matrix = np.dot(user_vector, item_vector.T)
print(predicted_matrix.shape)

X_train shape: (610, 8983)
(610, 8983)


In [13]:
# Function: predict_cf_rating
# Description: To return the predicted rating for a given user ID and movie ID
def predict_cf_rating(user_id, movie_id):
  '''
    Args:
        user_id: id of the user
        movie_id: id of the movie

    Output: returns predicted rating
  '''
  # Handling of users and movies part of cold-start problem, which cannot be predicted
  if user_id not in user_mapper or movie_id not in movie_mapper:
      return np.nan
  u_idx = user_mapper[user_id]
  m_idx = movie_mapper[movie_id]
  return predicted_matrix[u_idx, m_idx]

In [14]:
# Predict on the test set
test_ratings = R_test.copy()
test_ratings['cf_pred'] = test_ratings.apply(
    lambda row: predict_cf_rating(row['userId'], row['movieId']), axis=1
)

# Drop rows of cold-start users/movies
cf_eval = test_ratings.dropna(subset=['cf_pred'])

**Calculation of MSE and RMSE**

In [15]:
cf_mse = mean_squared_error(cf_eval['rating'], cf_eval['cf_pred'])
cf_rmse = np.sqrt(cf_mse)
print(f"Collaborative Filtering — MSE: {cf_mse:.4f}, RMSE: {cf_rmse:.4f}")

Collaborative Filtering — MSE: 9.0223, RMSE: 3.0037
